# 1.3 Verslo duomenys: klientų nutekėjimo prognozė su TensorFlow ir Decision Forests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbaranovskij/krepselisai/blob/main/notebooks/1_3_verslo_duomenys_TF_ir_DecisionForests.ipynb)

**Komanda:** Edvin Vilkanec, Gabriel Baranovskij · VILNIUSTECH

Dėstytojo pavyzdžiai *AdventureWorks DW using TensorFlow* ir *AdventureWorks DW using TensorFlow Decision Forests* iš įmonės duomenų saugyklos lentelės prognozuoja, ar klientas pirks dviratį. Čia tą patį metodą perdarome su **verslo duomenimis iš Kaggle**: **Telco Customer Churn** (7043 telekomunikacijų įmonės klientai). Prognozuojame, ar klientas **nutrauks sutartį** (`Churn`).

| Dalis | Modelis | Ką parodo |
|---|---|---|
| A | TensorFlow / Keras neuroninis tinklas | duomenis reikia paruošti: koduoti kategorijas, normalizuoti skaičius |
| B | Decision Forests (Gradient Boosted Trees) | modelis pats supranta kategorijas ir parodo, kurie požymiai svarbiausi |

In [ ]:
%pip install -q ydf

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

print("TensorFlow", tf.__version__)

## 1. Duomenys

Kaggle rinkinys *Telco Customer Churn* (originalas IBM). Kiekviena eilutė yra klientas: sutarties tipas, mokėjimo būdas, paslaugos, mėnesio mokestis, kiek mėnesių klientas su įmone (`tenure`) ir ar jis išėjo (`Churn`).

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

# TotalCharges kai kur tuščias (nauji klientai), paverčiame skaičiumi
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])

print(df.shape)
df.head()

In [ ]:
print(f"Išėjusių klientų dalis: {(df['Churn'] == 'Yes').mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
(df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
   .plot.bar(ax=ax[0], color="#12A36A", title="Išėjusių dalis pagal sutarties tipą, %"))
df.boxplot(column="tenure", by="Churn", ax=ax[1])
ax[1].set_title("Kiek mėnesių su įmone (tenure)"); plt.suptitle("")
plt.tight_layout(); plt.show()

Matyti, kad dažniausiai išeina klientai su **mėnesine sutartimi** ir **nauji klientai** (mažas `tenure`).

Abu modelius mokome ir tikriname su **ta pačia** 80 / 20 duomenų dalimi, kad palyginimas būtų sąžiningas.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Churn"])
print(len(train_df), "mokymui,", len(test_df), "testavimui")

## 2. A dalis: TensorFlow / Keras neuroninis tinklas

Neuroninis tinklas supranta tik skaičius, todėl:
1. **kategorinius** stulpelius (sutartis, mokėjimo būdas...) paverčiame 0/1 stulpeliais (*one-hot*, `pd.get_dummies`);
2. **skaitinius** stulpelius normalizuojame Keras `Normalization` sluoksniu (vidurkis 0, nuokrypis 1);
3. tinklas: du `Dense` sluoksniai su ReLU, `Dropout` prieš persimokymą ir išvesties neuronas su **sigmoid**, kuris duoda tikimybę, kad klientas išeis.

In [ ]:
kategoriniai = [c for c in df.columns if df[c].dtype == "object" and c != "Churn"]

def paruosti(d):
    X = pd.get_dummies(d.drop(columns=["Churn"]), columns=kategoriniai)
    return X, (d["Churn"] == "Yes").astype("float32").values

X_train, y_train = paruosti(train_df)
X_test, y_test = paruosti(test_df)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train, X_test = X_train.astype("float32").values, X_test.astype("float32").values
print("Požymių po one-hot:", X_train.shape[1])

norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

modelis = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    norm,
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelis.compile(optimizer="adam", loss="binary_crossentropy",
                metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
modelis.summary()

In [ ]:
istorija = modelis.fit(
    X_train, y_train, validation_split=0.2, epochs=40, batch_size=64, verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

h = istorija.history
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(h["loss"], label="mokymas"); ax[0].plot(h["val_loss"], label="validacija"); ax[0].set_title("Nuostoliai (loss)")
ax[1].plot(h["accuracy"], label="mokymas"); ax[1].plot(h["val_accuracy"], label="validacija"); ax[1].set_title("Tikslumas")
for a in ax: a.set_xlabel("epocha"); a.legend()
plt.tight_layout(); plt.show()

nn_loss, nn_acc, nn_auc = modelis.evaluate(X_test, y_test, verbose=0)
print(f"Keras testavimo tikslumas: {nn_acc:.3f}, AUC: {nn_auc:.3f}")

## 3. B dalis: Decision Forests

Dėstytojo notebook'e naudojama **TensorFlow Decision Forests**. Google tą pačią biblioteką dabar vysto pavadinimu **YDF** (*Yggdrasil Decision Forests*), tai yra TF-DF variklis su paprastesne sąsaja, todėl naudojame jį.

Skirtumai nuo neuroninio tinklo:
- **duomenų ruošti nereikia**: modelis pats supranta tekstines kategorijas ir nenormalizuotus skaičius;
- **Gradient Boosted Trees**: šimtai mažų sprendimų medžių, kiekvienas taiso ankstesnių klaidas;
- modelis parodo **požymių svarbą**, todėl rezultatą lengva paaiškinti verslui.

In [ ]:
import ydf

gbt = ydf.GradientBoostedTreesLearner(label="Churn").train(train_df)
ivertinimas = gbt.evaluate(test_df)
df_acc = ivertinimas.accuracy
print(f"Decision Forests testavimo tikslumas: {df_acc:.3f}")

In [ ]:
svarba = gbt.variable_importances()
raktas = "INV_MEAN_MIN_DEPTH" if "INV_MEAN_MIN_DEPTH" in svarba else next(iter(svarba))
top = svarba[raktas][:10]

plt.figure(figsize=(8, 4))
plt.barh([v for _, v in top][::-1], [s for s, _ in top][::-1], color="#12A36A")
plt.title(f"Svarbiausi požymiai ({raktas})"); plt.tight_layout(); plt.show()

## 4. Palyginimas

In [ ]:
rez = pd.DataFrame({"Modelis": ["Keras neuroninis tinklas", "Decision Forests (GBT)"],
                    "Tikslumas": [nn_acc, df_acc]})
ax = rez.plot.bar(x="Modelis", y="Tikslumas", legend=False, rot=0, color=["#2f6fdb", "#12A36A"], figsize=(7, 3.5))
ax.set_ylim(0.5, 1); ax.set_title("Testavimo tikslumas")
for i, v in enumerate(rez["Tikslumas"]): ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()
rez

## 5. Išvados

- Abu modeliai klientų nutekėjimą prognozuoja panašiu tikslumu (apie 0,8). Su **lentelių duomenimis** sprendimų miškai paprastai veikia ne prasčiau nei neuroninis tinklas.
- **Decision Forests** buvo paprastesnis: nereikėjo jokio duomenų paruošimo, o požymių svarba iškart parodo, kas lemia išėjimą (sutarties tipas, `tenure`, mokesčiai).
- **Neuroniniam tinklui** reikėjo one-hot kodavimo ir normalizavimo, bet jis lankstesnis, kai duomenų daug arba jie nestruktūruoti.
- Tas pats metodas kaip AdventureWorks pavyzdyje: verslo lentelė → požymiai → klasifikacija „taip / ne“.